# =========================================================
# CATBOOST MODEL — HDB RESALE PRICE PREDICTION
# =========================================================

In [2]:
# =========================================================
# 0. IMPORT LIBRARIES
# =========================================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score

## Step 1: Load and Clean Data

Three sequential cleaning operations:

1. **Drop nulls** — rows with any missing value are removed (primarily affects amenity distance columns where geocoding returned no result: 157,821 → 114,147, dropped 43,674)
2. **Drop duplicates** — `1 ROOM` (41 rows) and `MULTI-GENERATION` (43 rows) excluded due to insufficient sample size (< 50 each, < 0.1% of data); outside the typical buyer use case (114,147 → 114,071, dropped 76)
3. **Drop rare flat types** — `1 ROOM` (41 rows) and `MULTI-GENERATION` (43 rows) excluded due to insufficient sample size (< 50 each, < 0.1% of data); outside the typical buyer use case (114,071 → 113,981, dropped 90)
4. **Drop 2020** — excluded due to the COVID-19 structural break; abnormal market conditions unrepresentative of typical pricing relationships (113,981 → 96,736, dropped 17,245)

In [4]:
file_path = r"C:\Users\Jing Xuan\Desktop\Y3S2\DSE3101\DSE3101 Group Project\hdb_with_amenities_macro.csv"
df_raw = pd.read_csv(file_path)
print(f"Initial shape: {df_raw.shape}")

df = df_raw.dropna()
print(f"After dropping nulls: {df.shape} (dropped {len(df_raw) - len(df)})")

# Remove duplicate rows to avoid repeated transactions affecting model training
n_before = len(df)
df = df.drop_duplicates()
print(f"After dropping duplicates: {df.shape} (dropped {n_before - len(df)})")

# Excluded due to insufficient sample size (fewer than 50 transactions each),
# representing less than 0.1% of the dataset and falling outside the typical buyer use case.
mask_flat = ~df["flat_type"].isin(["1 ROOM", "MULTI-GENERATION"])
n_before = len(df)
df = df[mask_flat]
print(f"After dropping 1 ROOM and MULTI-GENERATION: {df.shape} (dropped {n_before - len(df)})")

# 2020 is excluded due to the COVID-19 structural break which caused abnormal market conditions
# unrepresentative of typical pricing relationships.
df["year_temp"] = pd.to_datetime(df["month"]).dt.year
n_before = len(df)
df = df[df["year_temp"] != 2020].drop(columns="year_temp")
print(f"After dropping 2020: {df.shape} (dropped {n_before - len(df)})")

df['log_resale_price_real'] = np.log(df['resale_price_real'])  # Log-transform target — preserves resale_price_real for metric computation

# Count of primary schools within 1 km — derived from pipe-separated names already in the dataset.
# Captures school density independently of nearest-school distance.
df['num_primary_1km'] = df['primary_schools_1km'].apply(
    lambda x: len(x.split('|')) if pd.notna(x) and x != '' else 0
)
print(f"num_primary_1km — mean: {df['num_primary_1km'].mean():.2f}, max: {df['num_primary_1km'].max()}")

# Count of primary schools within 1 km
df["num_primary_1km"] = df["primary_schools_1km"].apply(
    lambda x: len(x.split("|")) if pd.notna(x) and x != "" else 0
)

print(f"num_primary_1km — mean: {df['num_primary_1km'].mean():.2f}, max: {df['num_primary_1km'].max()}")

Initial shape: (157821, 37)
After dropping nulls: (114147, 37) (dropped 43674)
After dropping duplicates: (114071, 37) (dropped 76)
After dropping 1 ROOM and MULTI-GENERATION: (113981, 37) (dropped 90)
After dropping 2020: (96736, 37) (dropped 17245)
num_primary_1km — mean: 2.99, max: 9
num_primary_1km — mean: 2.99, max: 9


## Step 2: Feature Engineering

Five new columns created from existing raw columns:

| New column | Source | Logic |
|-----------|--------|-------|
| `remaining_lease_years` | `remaining_lease` | Parse `"61 years 04 months"` → `61.33` (handles missing months) |
| `floor_category` | `storey_range` | Extract lower floor from `"10 TO 12"` → `10`; bucket as Low (1–5), Mid (6–12), High (13+) |
| `year` | `month` | Integer year — used for stratification only, **not** a model feature |
| `num_schools` | `primary_schools_1km` | Count number of primary schools within 1km |
| `num_parks` | `parks_1km` | Count number of parks within 1km |

**Target variable:** `log_resale_price_real` — using RPI-adjusted prices means the model captures structural flat-level pricing drivers rather than market-wide appreciation, since `year` is excluded as a feature.


In [5]:
# =========================================================
# STEP 2: FEATURE ENGINEERING
# Match OLS baseline:
# - remaining_lease_years
# - floor_category
# - year
# - target = log_resale_price_real
# =========================================================
def parse_remaining_lease(s):
    match = re.match(r"(\d+) years?(?: (\d+) months?)?", str(s))
    if match:
        years = int(match.group(1))
        months = int(match.group(2)) if match.group(2) else 0
        return round(years + months / 12, 2)
    return np.nan

df["remaining_lease_years"] = df["remaining_lease"].apply(parse_remaining_lease)

# Extract lower floor number from storey_range (e.g., "10 TO 12" → 10)
df["floor_lower"] = df["storey_range"].str.extract(r"^(\d+)").astype(int)
df["floor_category"] = pd.cut(
    df["floor_lower"],
    bins=[0, 5, 12, float("inf")],
    labels=["Low", "Mid", "High"],
    right=True
)

# Year for stratification only — will NOT be used as a model feature
df["year"] = pd.to_datetime(df["month"]).dt.year


# Count number of primary schools within 1km
df["num_schools_1km"] = df["primary_schools_1km"].str.count(r"\|") + 1

# Count number of parks within 1km
df["num_parks_1km"] = df["parks_1km"].str.count(r"\|") + 1



# Target variable: log_resale_price_real
# RPI-adjusted prices are used so the model learns purely from flat characteristics
# rather than market-wide price appreciation over time, since year is not a model feature.
target = "log_resale_price_real"

print(df[["remaining_lease", "remaining_lease_years", "storey_range",
          "floor_lower", "floor_category", "year", 
          "primary_schools_1km", "num_schools_1km", "parks_1km", "num_parks_1km"]].head(10))

          remaining_lease  remaining_lease_years storey_range  floor_lower  \
23333   64 years 01 month                  64.08     01 TO 03            1   
23334   64 years 01 month                  64.08     07 TO 09            7   
23335            59 years                  59.00     04 TO 06            4   
23336  58 years 02 months                  58.17     04 TO 06            4   
23337   58 years 01 month                  58.08     01 TO 03            1   
23338  64 years 02 months                  64.17     07 TO 09            7   
23339  58 years 02 months                  58.17     04 TO 06            4   
23340   59 years 01 month                  59.08     04 TO 06            4   
23341            64 years                  64.00     07 TO 09            7   
23342  54 years 04 months                  54.33     04 TO 06            4   

      floor_category  year                                primary_schools_1km  \
23333            Low  2021  ANG MO KIO PRIMARY SCHOOL|MAYFLO

## Step 3: Stratified Train/Validation Split (80/20)

**80/20 ratio:** With ~97,000 transactions, yields ~77,000 training and ~19,000 validation rows. The training set is large enough for stable estimation; the validation set is large enough for statistically reliable RMSE and linlin loss estimates. A 70/30 split is only preferred for datasets of a few thousand rows.

**Stratification key: `town + flat_type + year`** — three reasons:
1. **Town** — 26 HDB towns with substantially different price levels; random sampling could over-represent expensive towns in training.
2. **Flat type** — Highly imbalanced (4 ROOM ~42% vs 2 ROOM ~2%); stratifying prevents minority types being underrepresented.
3. **Year** — Despite RPI adjustment, residual structural differences exist across years (post-COVID demand surge, cooling measures); ensures proportional representation of every year in both sets.

In [6]:
# =========================================================
# STEP 3: STRATIFIED TRAIN / VALIDATION SPLIT (80/20)
# from development set only (2021–2025)
# =========================================================

df["strat_key"] = (
    df["town"].astype(str) + "_" +
    df["flat_type"].astype(str) + "_" +
    df["year"].astype(str)
)

strat_counts = df["strat_key"].value_counts()
valid_keys = strat_counts[strat_counts >= 2].index
n_before = len(df)
df = df[df["strat_key"].isin(valid_keys)].copy()
print(f"Dropped {n_before - len(df)} rows with singleton strat_key combinations. Remaining: {len(df):,}")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["strat_key"]
)

print(f"Train size: {len(train_df):,} | Validation size: {len(val_df):,}")

print("\nYear distribution (%):")
train_year = train_df["year"].value_counts(normalize=True).sort_index().rename("Train %")
val_year = val_df["year"].value_counts(normalize=True).sort_index().rename("Validation %")
year_dist = pd.concat([train_year, val_year], axis=1)
print(year_dist.map(lambda x: f"{x:.2%}"))

print("\nFlat type distribution (%):")
train_flat = train_df["flat_type"].value_counts(normalize=True).rename("Train %")
val_flat = val_df["flat_type"].value_counts(normalize=True).rename("Validation %")
flat_dist = pd.concat([train_flat, val_flat], axis=1)
print(flat_dist.map(lambda x: f"{x:.2%}"))

Dropped 7 rows with singleton strat_key combinations. Remaining: 96,729
Train size: 77,383 | Validation size: 19,346

Year distribution (%):
     Train % Validation %
year                     
2021  21.91%       21.91%
2022  19.93%       19.98%
2023  18.78%       18.77%
2024  20.62%       20.61%
2025  18.76%       18.73%

Flat type distribution (%):
          Train % Validation %
flat_type                     
4 ROOM     41.87%       41.88%
3 ROOM     25.56%       25.54%
5 ROOM     23.60%       23.60%
EXECUTIVE   6.75%        6.77%
2 ROOM      2.22%        2.21%


## Step 4: Feature Preparation

We prepare the feature matrices and target variables for model training.

- **Numerical features** include proximity measures (e.g., distance to MRT, hawker centres) and lease characteristics  
- **Categorical features** include flat type, town, and floor category  

The target variable is the **log-transformed real resale price**, which helps:
- Stabilise variance  
- Improve model performance  
- Reduce skewness in price distribution  

We also retain the original (non-log) resale price for evaluation in dollar terms.

Categorical features are explicitly passed to CatBoost, which handles encoding internally.

In [8]:
# =========================================================
# STEP 4: PREPARE FEATURES FOR CATBOOST
#
# continuous_features = [
#   remaining_lease_years, nearest_train_dist_m, dist_nearest_hawker_m,
#   dist_nearest_primary_m, num_primary_1km, dist_nearest_park_m,
#   dist_nearest_sportsg_m, dist_nearest_mall_m, dist_nearest_healthcare_m
# ]
# categorical_features = [flat_type, town, floor_category]
#
# Excluded:
# - floor_area_sqm
# - dist_cbd_m
# =========================================================
feature_cols = [
    "remaining_lease_years",
    "nearest_train_dist_m",
    "dist_nearest_hawker_m",
    "dist_nearest_primary_m",
    "num_primary_1km",
    "dist_nearest_park_m",
    "dist_nearest_sportsg_m",
    "dist_nearest_mall_m",
    "dist_nearest_healthcare_m",
    "flat_type",
    "town",
    "floor_category"
]

categorical_features = ["flat_type", "town", "floor_category"]

X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()

y_train = train_df[target].copy()
y_val = val_df[target].copy()

# Actual prices retained for evaluation in SGD
y_train_actual = train_df["resale_price_real"].copy()
y_val_actual = val_df["resale_price_real"].copy()

cat_feature_indices = [X_train.columns.get_loc(col) for col in categorical_features]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Categorical feature indices:", cat_feature_indices)
print("Categorical features:", categorical_features)

X_train shape: (77383, 12)
X_val shape: (19346, 12)
Categorical feature indices: [9, 10, 11]
Categorical features: ['flat_type', 'town', 'floor_category']


## Step 5: Model Training (CatBoost)

We train a **CatBoost Regressor**, which is well-suited for tabular data and handles categorical variables natively.

Key settings:
- Loss function: RMSE  
- Early stopping: stops training if validation performance does not improve  
- Validation set: used to monitor performance and prevent overfitting  

The model is trained on the training set and evaluated on the validation set during training.  
The best iteration (based on validation performance) is automatically selected.

In [28]:
# =========================================================
# STEP 5: FIT CATBOOST MODEL
# Since this is for fair comparison with OLS, we keep one train/test split.
# We use a validation split inside the training data for early stopping.
# =========================================================
train_sub_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["flat_type"]
)

X_train_sub = train_sub_df[feature_cols].copy()
y_train_sub = train_sub_df[target].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target].copy()

cat_feature_indices_sub = [X_train_sub.columns.get_loc(col) for col in categorical_features]

cat_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    early_stopping_rounds=100,
    verbose=200
)

cat_model.fit(
    X_train_sub,
    y_train_sub,
    cat_features=cat_feature_indices_sub,
    eval_set=(X_val, y_val),
    use_best_model=True
)

print("Best iteration:", cat_model.get_best_iteration())
print("Best validation score:", cat_model.get_best_score())

0:	learn: 0.3057228	test: 0.3066553	best: 0.3066553 (0)	total: 66.8ms	remaining: 3m 20s
200:	learn: 0.0798735	test: 0.0827976	best: 0.0827976 (200)	total: 15.1s	remaining: 3m 30s
400:	learn: 0.0681976	test: 0.0714811	best: 0.0714811 (400)	total: 30.5s	remaining: 3m 17s
600:	learn: 0.0630686	test: 0.0666461	best: 0.0666461 (600)	total: 45.7s	remaining: 3m 2s
800:	learn: 0.0601589	test: 0.0642088	best: 0.0642088 (800)	total: 1m 1s	remaining: 2m 48s
1000:	learn: 0.0582359	test: 0.0626485	best: 0.0626485 (1000)	total: 1m 16s	remaining: 2m 32s
1200:	learn: 0.0567385	test: 0.0615713	best: 0.0615713 (1200)	total: 1m 32s	remaining: 2m 18s
1400:	learn: 0.0555743	test: 0.0608132	best: 0.0608132 (1400)	total: 1m 48s	remaining: 2m 3s
1600:	learn: 0.0546211	test: 0.0602191	best: 0.0602191 (1600)	total: 2m 3s	remaining: 1m 48s
1800:	learn: 0.0538290	test: 0.0597760	best: 0.0597760 (1800)	total: 2m 19s	remaining: 1m 32s
2000:	learn: 0.0531334	test: 0.0594105	best: 0.0594105 (2000)	total: 2m 35s	remai

## Step 6: Generating Predictions

After training, predictions are generated for:

- **Training set** (optional, for diagnostics)  
- **Validation set** (for model selection)  

Since the model is trained on log-transformed prices, predictions are exponentiated to obtain values in actual dollar terms.

In [29]:
# =========================================================
# STEP 6: GENERATE PREDICTIONS
# Predictions made in log space, then converted back to SGD
# =========================================================
y_test_pred_log = cat_model.predict(X_test)
y_test_pred = np.exp(y_test_pred_log)

# Optional: train predictions
y_train_pred_log = cat_model.predict(X_train)
y_train_pred = np.exp(y_train_pred_log)

## Step 7: Model Evaluation

We evaluate model performance using three metrics:

1. **RMSE (Root Mean Squared Error)**  
   - Measures prediction accuracy in dollar terms  
   - Penalises large errors more heavily  

2. **R² (Coefficient of Determination)**  
   - Measures proportion of variance explained by the model  

3. **Lin-Lin Loss (Asymmetric Loss)**  
   - Penalises **underprediction more heavily than overprediction**  
   - Reflects real-world risk (e.g., underestimating housing prices may lead to budget constraints)  

Evaluation is conducted in two stages:

- **Validation set** → used for model tuning and selection  
- **Test set (2026)** → used for final, unbiased performance evaluation  

The test set is not used during model development to ensure a fair assessment of generalisation performance.

In [30]:
# =========================================================
# STEP 7: EVALUATION
# Match OLS metrics:
# - RMSE
# - asymmetric Lin-Lin loss
# - 80% interval coverage (using conformal interval instead of OLS formula)
# =========================================================
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    """
    Asymmetric linear loss penalising underprediction more heavily than overprediction.
    underpredict_weight=2.0 means predicted < actual is penalised at 2x.
    Overprediction is penalised at 1x.
    """
    errors = np.array(y_true) - np.array(y_pred)  # positive = underprediction
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

test_rmse = rmse(y_test_actual, y_test_pred)
test_linlin = linlin_loss(y_test_actual, y_test_pred, underpredict_weight=2.0)
test_r2 = r2_score(y_test_actual, y_test_pred)

print("=== CATBOOST MODEL EVALUATION ===")
print(f"RMSE:          ${test_rmse:,.0f}")
print(f"R²:            {test_r2:.3f}")
print(f"Linlin Loss:   ${test_linlin:,.0f} (underpredict weight = 2.0)")

=== CATBOOST MODEL EVALUATION ===
RMSE:          $41,236
R²:            0.963
Linlin Loss:   $43,361 (underpredict weight = 2.0)
